In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import numpy as np

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
   # -----------------------------
# CBAM MODULE
# -----------------------------
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(in_planes, in_planes // ratio, 1),
            nn.ReLU(),
            nn.Conv2d(in_planes // ratio, in_planes, 1)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        max, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg, max], dim=1)
        return self.sigmoid(self.conv(x))


class CBAM(nn.Module):
    def __init__(self, in_planes):
        super().__init__()
        self.ca = ChannelAttention(in_planes)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x



In [ ]:
# -----------------------------
# ConvNeXt + CBAM Model
# -----------------------------
class ConvNeXt_CBAM_Model(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = timm.create_model("convnext_tiny", pretrained=True, features_only=True)
        self.out_ch = self.backbone.feature_info[-1]['num_chs']
        self.cbam = CBAM(self.out_ch)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.out_ch, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)[-1]
        x = self.cbam(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

In [ ]:
# -----------------------------
# Transforms (Data Augmentation)
# -----------------------------
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
# -----------------------------
# Dataset Paths
# -----------------------------
train_dir = 'dataset/train'
test_dir = 'dataset/test'

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)


In [ ]:

# -----------------------------
# Weighted Sampler for Class Balancing
# -----------------------------
targets = [label for _, label in train_dataset]
class_counts = np.bincount(targets)
class_weights = 1. / class_counts
sample_weights = [class_weights[label] for label in targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


In [ ]:

# -----------------------------
# Data Loaders
# -----------------------------
train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes  # ['healthy', 'onychomycosis', 'psoriasis']


In [ ]:

# -----------------------------
# Initialize Model, Loss, Optimizer
# -----------------------------
model = ConvNeXt_CBAM_Model(num_classes=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)



In [ ]:
# -----------------------------
# Training Function
# -----------------------------
def train(model, loader):
    model.train()
    running_loss, correct = 0.0, 0
    for inputs, labels in tqdm(loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    acc = 100. * correct / len(loader.dataset)
    return running_loss / len(loader), acc



In [ ]:
# -----------------------------
# Evaluation Function (with metrics)
# -----------------------------
def evaluate_with_metrics(model, loader, class_names, dataset_name="Test"):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # ---------------------
    # Confusion Matrix
    # ---------------------
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n{dataset_name} Confusion Matrix:\n", cm)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{dataset_name} Confusion Matrix")
    plt.show()

    # ---------------------
    # Classification Report & Bar Plots
    # ---------------------
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    print(f"\n{dataset_name} Classification Report:\n", classification_report(y_true, y_pred, target_names=class_names))

    # Bar Plots for Precision, Recall, F1-Score
    metrics = ['precision', 'recall', 'f1-score']
    for m in metrics:
        scores = [report[cls][m] for cls in class_names]
        plt.figure(figsize=(6, 4))
        sns.barplot(x=class_names, y=scores)
        plt.title(f"{dataset_name} {m.capitalize()} per Class")
        plt.ylim(0, 1)
        plt.ylabel(m.capitalize())
        plt.show()

    # ---------------------
    # Misclassified Image Visualization
    # ---------------------
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1)

            for i in range(len(preds)):
                if preds[i] != y[i].to(device):
                    plt.imshow(x[i].permute(1, 2, 0).cpu())
                    plt.title(f"True: {class_names[y[i]]}, Pred: {class_names[preds[i]]}")
                    plt.axis('off')
                    plt.show()


In [ ]:
# -----------------------------
# Train & Evaluate
# -----------------------------
epochs = 50
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

# Final Evaluation on Training and Test Data
evaluate_with_metrics(model, train_loader, class_names, dataset_name="Train")
evaluate_with_metrics(model, test_loader, class_names, dataset_name="Test")
